# Anomaly detection

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## The idea in one look
An **anomaly** (or outlier) is a sample unlike the others: a fraudulent transaction, a faulty machine, a network attack. Usually there are no labels and anomalies are rare, so the problem is unsupervised.

## Methods
| Method | Idea | scikit-learn |
|---|---|---|
| Statistical (Z-score, IQR) | Distance from the mean or the quartiles | By hand |
| Isolation Forest | A strange point is **isolated quickly** by random cuts | `IsolationForest` |
| LOF | Density around a point compared with its neighbors | `LocalOutlierFactor` |
| Model-based | Low probability under a GMM | `GaussianMixture.score_samples` |
| One-Class SVM | A boundary around the normal data | `OneClassSVM` |

## The IQR rule (simple and widely used)
First quartile $Q_1$, third quartile $Q_3$, $IQR=Q_3-Q_1$. Points outside $[Q_1-1.5IQR,\ Q_3+1.5IQR]$ are anomalies (the dots in a box plot).

| Data | $Q_1$ | $Q_3$ | IQR | Allowed range | Anomaly |
|---|---|---|---|---|---|
| 10, 12, 12, 13, 14, 15, 40 | 12 | 14.5 | 2.5 | [8.25, 18.25] | 40 |



In [ ]:
import numpy as np

x = np.array([10, 12, 12, 13, 14, 15, 40])
q1, q3 = np.percentile(x, [25, 75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1}, Q3={q3}, IQR={iqr}, allowed=[{low}, {high}]")
print("anomalies:", x[(x < low) | (x > high)])
z = (x - x.mean()) / x.std()
print("z-scores:", z.round(2))



## Isolation Forest
It builds random trees: a random feature and a random cut. A normal point sits in the crowd and needs **many cuts** to be isolated; an anomaly is alone after **a few cuts**. The anomaly score comes from the **average isolation depth** over all trees.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

rng = np.random.default_rng(0)
normal = rng.normal(0, 1, (300, 2))
odd = rng.uniform(-6, 6, (15, 2))
X = np.r_[normal, odd]

iso = IsolationForest(contamination=0.05, random_state=0).fit(X)
lab_iso = iso.predict(X)                 # -1 = anomaly
lab_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05).fit_predict(X)
print("Isolation Forest found", np.sum(lab_iso == -1), "anomalies;", np.sum(lab_iso[300:] == -1), "of the 15 planted ones")
print("LOF found", np.sum(lab_lof == -1), "anomalies;", np.sum(lab_lof[300:] == -1), "of the 15 planted ones")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, lab, t in [(ax[0], lab_iso, "Isolation Forest"), (ax[1], lab_lof, "LOF")]:
    a.scatter(X[:, 0], X[:, 1], c=np.where(lab == -1, "red", "lightgray"), s=12); a.set_title(t)
plt.show()



> The `contamination` parameter sets the share of anomalies and so the threshold. If you do not know it, use `score_samples` instead of labels and inspect the highest scores.